# Hardseal-Occam — ARC Prize 2026 ARC-AGI-3 (Kernel v11 — REAL AGENT)

**Bundle:** vendored as Kaggle Dataset `ricoallen/hardseal-occam-vendored` (mounted at `/kaggle/input/hardseal-occam-vendored/`).
- `occam_upstream/` — Sean Donahoe's [Occam](https://github.com/g-baskin/occam) ARC-AGI-3 solver (MIT, vendored verbatim at commit `e3be26a`)
- `hardseal_trace/` — `hardseal-trace` v0.2.0 cryptographic reasoning-trace primitive (Apache-2.0, vendored)
- `hardseal_occam/` — wrapper bridging Occam's `EventEmitter` to `SealedReasoningTracer` (CC0-1.0 OR MIT, this repo)

**v11 structural fix:** ARC-AGI-3 Kaggle is a **Notebook Inference Server agent harness** competition, not a static-file competition. Kaggle auto-generates `submission.parquet` from the agent's gameplay scorecard. v9/v10's stub-and-exit + user-written submission.parquet was always going to fail. v11 drops Path A entirely and runs an unconditional agent loop every save and every rerun, per `Arcade(operation_mode=OperationMode.COMPETITION)`.

Per ARC Prize 2026 contest rules, all wrapper code is `CC0-1.0 OR MIT`. See `NOTICE.md` and `LICENSES/` in the dataset for the full provenance map.

Source repo (mirror): https://github.com/ricojallen37-sketch/hardseal-occam


## Cell 1 — Install arc-agi SDK + nest_asyncio + set up vendored paths

Unconditional. Runs every save and every rerun. `enable_internet=false` in kernel metadata; pip install will only succeed when Kaggle's offline-but-PyPI-cached wheels resolve, otherwise we fall back to whatever is already on the image and let the import below decide.


In [ ]:
import os
import sys
import subprocess
import traceback
from pathlib import Path

# Vendored code mounted from the Kaggle Dataset at /kaggle/input/hardseal-occam-vendored/
DATASET_ROOT = Path('/kaggle/input/hardseal-occam-vendored')
if not DATASET_ROOT.exists():
    # Local-run fallback: load from the contest repo dir alongside this notebook
    DATASET_ROOT = Path('.').resolve()

# Add vendored paths to sys.path so imports resolve before any pip install runs
for p in [str(DATASET_ROOT), str(DATASET_ROOT / 'occam_upstream')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# arc-agi SDK + nest_asyncio. With enable_internet=false this best-effort install
# will only succeed if Kaggle's image already has the wheel cached or arc-agi is
# pre-installed in the rerun environment. If it fails the import below will surface
# a clear error.
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'arc-agi==0.9.8', 'nest_asyncio'],
    )
except Exception as _e:
    print(f'pip install arc-agi/nest_asyncio failed (may already be present): {_e}')

# Allow asyncio.run inside Jupyter's already-running event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception as _e:
    print(f'nest_asyncio.apply() failed: {_e}')

print(f'DATASET_ROOT={DATASET_ROOT}')
print(f'Python {sys.version.split()[0]}')
print(f'sys.path[:3]: {sys.path[:3]}')


## Cell 2 — Provenance fingerprint (license map + content SHAs)


In [ ]:
import hashlib

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

fingerprint_files = [
    'occam_upstream/solver/orchestrator.py',
    'occam_upstream/solver/benchmark.py',
    'hardseal_trace/__init__.py',
    'hardseal_occam/tracer_callback.py',
    'hardseal_occam/runner.py',
    'LICENSE',
    'NOTICE.md',
    'LICENSES/MIT-occam.txt',
    'LICENSES/Apache-2.0-hardseal-trace.txt',
]
fingerprints = {}
for k in fingerprint_files:
    p = DATASET_ROOT / k
    fingerprints[k] = _sha256(p) if p.exists() else 'MISSING'

for k, v in fingerprints.items():
    print(f'  {v[:16]:16s}  {k}')


## Cell 3 — Define `HardsealOccamAgent` (per-game adapter wrapping Occam's orchestrator + hardseal-trace callback)

Per-game agent. Constructed with `(env, skip_deepcopy=True)`; `agent.run()` plays one game while emitting hardseal-trace sealed reasoning logs through `HardsealOccamCallback`. Per-game chains are appended to `_AGENT_RESULTS` so we can summarise + verify after the unconditional loop completes.


In [ ]:
import asyncio
import time
import uuid

from solver.environments import ArcEnvAdapter
from solver.orchestrator import GameOrchestrator
from solver.events import EventEmitter

from hardseal_occam.tracer_callback import HardsealOccamCallback
from hardseal_trace import verify_chain

# Run-level identifiers + HMAC key for hardseal-trace
RUN_UUID = str(uuid.uuid4())
HMAC_KEY = b'hardseal-occam-v0.4-hmac-key'
MAX_ACTIONS = 500_000  # matches occam viewer/cli.py default; ample for eval

# Shared callback collects per-game sealed reasoning chains across the whole run
_CALLBACK = HardsealOccamCallback(hmac_key=HMAC_KEY, run_uuid=RUN_UUID)
_AGENT_RESULTS = []  # appended one entry per game by HardsealOccamAgent.run()


class HardsealOccamAgent:
    """Per-game agent that wraps Occam's GameOrchestrator with hardseal-trace.

    Constructed once per env. ``run()`` plays the single game associated with this
    env, emits sealed reasoning traces through the shared ``_CALLBACK``, and records
    the result into ``_AGENT_RESULTS``.
    """

    def __init__(self, env, skip_deepcopy: bool = True, max_actions: int = MAX_ACTIONS):
        # ``env`` is the raw arc_agi env handle returned from ``arcade.make(game_id)``.
        # Wrap it in Occam's ArcEnvAdapter — the adapter Occam expects internally.
        self._raw_env = env
        self._adapter = ArcEnvAdapter(env)
        self._skip_deepcopy = skip_deepcopy
        self._max_actions = max_actions
        # Game info (game_id, title, baseline_actions, available_actions) lives on
        # the env_info object surfaced by arc.get_environments(). The caller stashes
        # it on the env via ``env._game_info`` before constructing the agent so the
        # adapter has access without a second SDK round-trip.
        self._game_info = getattr(env, '_game_info', None)

    def run(self) -> dict:
        """Synchronous entry point. Plays one game; returns a result dict."""
        events = EventEmitter(callback=_CALLBACK)
        orchestrator = GameOrchestrator(
            max_actions_per_level=self._max_actions,
            skip_deepcopy=self._skip_deepcopy,
            event_callback=events.callback,
        )
        baselines = (
            list(self._game_info.baseline_actions)
            if self._game_info is not None and hasattr(self._game_info, 'baseline_actions')
            else []
        )
        t0 = time.time()
        # nest_asyncio.apply() ran in Cell 1 — asyncio.run is safe even inside Jupyter.
        result = asyncio.run(orchestrator.play_game(self._adapter, baselines))
        wall_s = time.time() - t0
        if self._game_info is not None:
            result.update({
                'game_id': getattr(self._game_info, 'game_id', None),
                'title': getattr(self._game_info, 'title', None),
                'total_levels': len(baselines),
                'time_s': round(wall_s, 1),
            })
        _AGENT_RESULTS.append(result)
        gid = result.get('game_id', '<unknown>')
        score = result.get('game_score', 0.0)
        levels = result.get('levels_completed', 0)
        print(f'  [agent] game={gid} score={score:.4f} levels={levels} wall={wall_s:.1f}s')
        return result


## Cell 4 — Unconditional agent loop (the v11 structural fix)

`Arcade(operation_mode=OperationMode.COMPETITION)` → `arc.get_environments()` (try/except, empty list on failure) → for each game: `arc.make(game_id)` → `HardsealOccamAgent(env, skip_deepcopy=True)` → `agent.run()` wrapped in try/except so a single bad game doesn't crash the run. **No save-mode gate. No `KAGGLE_IS_COMPETITION_RERUN` check. No `OPERATION_MODE` env-var gate. No user-written `submission.parquet`.** Kaggle auto-generates the parquet from the agent's gameplay scorecard.


In [ ]:
import arc_agi
from arc_agi import OperationMode

# Explicit COMPETITION mode (Kaggle's rerun infra forces this server-side, but
# explicit-is-better-than-implicit and matches the public docs at
# docs.arcprize.org/toolkit/competition_mode).
arc = arc_agi.Arcade(operation_mode=OperationMode.COMPETITION)

# Get the games available in the current environment (public set at save-time,
# hidden set at Kaggle private rerun).
try:
    games = arc.get_environments()
except Exception as _e:
    print(f'arc.get_environments() failed: {_e}')
    games = []

print(f'arc.get_environments() -> {len(games)} games')

# Run the agent through each game with hardseal-trace reasoning logs. Per-game
# try/except so a single failed game doesn't short-circuit the loop (Competition
# Mode resilience requirement).
_run_t0 = time.time()
for game_info in games:
    try:
        env = arc.make(game_info.game_id)  # one make per env per Competition Mode invariants
        if env is None:
            print(f'  [skip] arc.make({game_info.game_id}) returned None')
            continue
        # Stash the env_info on the env so HardsealOccamAgent can access baseline_actions
        # without a second SDK round-trip.
        try:
            env._game_info = game_info
        except Exception:
            pass
        agent = HardsealOccamAgent(env, skip_deepcopy=True)
        agent.run()  # plays the game, emits sealed reasoning traces
    except Exception as _e:
        print(f'  [game-failed] {getattr(game_info, "game_id", "?")}: {type(_e).__name__}: {_e}')
        traceback.print_exc()
        continue  # next game

_run_wall_s = time.time() - _run_t0
print(f'\nAgent loop complete: {len(_AGENT_RESULTS)}/{len(games)} games run, wall={_run_wall_s:.1f}s')
print('Kaggle will auto-generate submission.parquet from the agent gameplay scorecard.')


## Cell 5 — Verify the sealed-trace chain (per-game)


In [ ]:
per_game_chains = _CALLBACK.export_per_game_chains()
validations = []
for i, gc in enumerate(per_game_chains):
    if not gc:
        validations.append((i, True, 'empty-chain'))
        continue
    try:
        ok, msg = verify_chain(gc, HMAC_KEY)
    except Exception as _e:
        ok, msg = False, f'exception: {_e}'
    validations.append((i, ok, msg))

all_valid = all(v[1] for v in validations)
total_traces = sum(len(gc) for gc in per_game_chains)
avg_pods = _CALLBACK.average_pods()

print(f'per-game chains: {len(per_game_chains)} ({sum(1 for v in validations if v[1])}/{len(per_game_chains)} valid)')
print(f'total traces:    {total_traces}')
print(f'avg PODS:        {avg_pods}')
print(f'all valid:       {all_valid}')
for i, ok, msg in validations[:5]:
    print(f'  game[{i}]: ({ok}, {msg})')
if len(validations) > 5:
    print(f'  ... +{len(validations) - 5} more')


## Cell 6 — Persist sealed-trace JSONL + summary as Kaggle output artifacts

These are diagnostic artifacts only; Kaggle reads the agent's gameplay scorecard (auto-generated `submission.parquet`) to score, not these files. Sealed traces preserve hardseal-trace's cryptographic reasoning-chain methodology contribution.


In [ ]:
import json

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATASET_ROOT / 'kaggle_run'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sealed traces (full chain across all games, flat)
flat_chain = _CALLBACK.export_chain()
chain_path = OUT_DIR / f'sealed_traces_{RUN_UUID}.jsonl'
with chain_path.open('w') as f:
    for trace in flat_chain:
        f.write(json.dumps(trace) + '\n')

# Aggregate stats from the per-game agent results recorded by the loop
games_solved = sum(1 for r in _AGENT_RESULTS if r.get('levels_completed', 0) > 0)
scores = [r.get('game_score', 0.0) for r in _AGENT_RESULTS]
mean_rhae_pct = (sum(scores) / len(scores) * 100) if scores else 0.0

summary = {
    'run_uuid': RUN_UUID,
    'agent': 'HardsealOccamAgent',
    'agent_version': '0.4.0',
    'kernel_version': 'v11',
    'wrapped_solver': 'occam-solver@e3be26a (MIT)',
    'trace_primitive': 'hardseal-trace@0.2.0 (Apache-2.0)',
    'wall_s': round(_run_wall_s, 1),
    'rhae_pct': mean_rhae_pct,
    'games_solved': games_solved,
    'n_games': len(_AGENT_RESULTS),
    'trace_count': len(flat_chain),
    'avg_pods': avg_pods,
    'all_chains_valid': all_valid,
    'per_game_validation': [
        {'game_idx': i, 'valid': ok, 'msg': msg, 'trace_count': len(per_game_chains[i])}
        for i, ok, msg in validations
    ],
    'callback_stats': _CALLBACK.stats(),
    'fingerprints': fingerprints,
}
summary_path = OUT_DIR / f'summary_{RUN_UUID}.json'
summary_path.write_text(json.dumps(summary, indent=2))

print(f'wrote {chain_path} ({len(flat_chain)} traces)')
print(f'wrote {summary_path}')
print(f'\nHardseal-Occam Kaggle Run v11')
print(f'  run_uuid:        {RUN_UUID}')
print(f'  RHAE:            {mean_rhae_pct:.4f}%')
print(f'  Games solved:    {games_solved}/{len(_AGENT_RESULTS)}')
print(f'  Wall time:       {_run_wall_s:.1f}s')
print(f'  All chains valid: {all_valid}')
print('\nNOTE: submission.parquet is NOT written by this kernel — Kaggle auto-generates it from the agent gameplay scorecard.')
